In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

kst = ZoneInfo("Asia/Seoul")

In [0]:

# =========================================================
# 1. Silver 기준 마지막 처리 시각 조회
# =========================================================
SLV_TABLE = "hive_metastore.demo_airstatus_silver.SLV_fact_air_quality"

row = (
    spark.table(SLV_TABLE)
    .selectExpr("MAX(dataTime) AS last_dataTime")
    .collect()
)

last_dataTime = row[0]["last_dataTime"]
last_dataTime = last_dataTime + timedelta(hours=9)

if last_dataTime is None:
    run_mode = "BACKFILL"
    print("[INFO] Silver table empty → BACKFILL required")

else:
    # ✅ Silver에서 나온 값은 KST 의미의 naive datetime
    last_dataTime_kst = last_dataTime.replace(tzinfo=kst)
    print(f"[INFO] last_dataTime (KST) : {last_dataTime_kst}")


[INFO] last_dataTime (KST) : 2026-05-08 00:00:00+09:00


In [0]:

    # =====================================================
    # 2. 현재 시각 (KST, timezone‑aware)
    # =====================================================
    now = datetime.now(kst)
    print(f"[INFO] current_time (KST)  : {now}")


[INFO] current_time (KST)  : 2026-05-08 13:52:28.655813+09:00


In [0]:

    # =====================================================
    # 3. DAILY 가능 여부 판단 (기존 로직 유지)
    # =====================================================
    daily_cutoff = now - timedelta(hours=22)
    print(f"[INFO] daily_cutoff (KST)  : {daily_cutoff}")


[INFO] daily_cutoff (KST)  : 2026-05-07 15:52:28.655813+09:00


In [0]:

    # ✅ 테스트용 출력 (네가 원한 부분)
    diff = now - last_dataTime_kst
    diff_hours = int(diff.total_seconds() // 3600)
    diff_minutes = int((diff.total_seconds() % 3600) // 60)

    print("-------------------------------------")
    print(f"now - last_dataTime = {diff_hours}시간 {diff_minutes}분")
    print("-------------------------------------")

    if last_dataTime_kst >= daily_cutoff:
        run_mode = "DAILY"
        print("[RESULT] DAILY job can catch up")
    else:
        run_mode = "BACKFILL"
        print("[RESULT] DAILY job cannot catch up → BACKFILL required")


-------------------------------------
now - last_dataTime = 13시간 52분
-------------------------------------
[RESULT] DAILY job can catch up


In [0]:

# =========================================================
# 4. Job Task Value로 결과 전달
# =========================================================
dbutils.jobs.taskValues.set(
    key="run_mode",
    value=run_mode
)

print(f"[INFO] run_mode set to: {run_mode}")


[INFO] run_mode set to: DAILY
